# Round 1 ancestry filtering

Fits a Mahalanobis-distance ellipsoid to a reference population cluster in 1000G's own PCA space (`submit_pca_r1.ipynb`'s output), then applies that same fitted ellipsoid to AoU's projected PCs to produce a keep-list. Same method as the earlier `ellopsoid_filter.py` prototype (6 PCs, chi2 90% threshold, CEU+GBR), now run against the real projected data instead of a chr22 validation subset.

Fit and threshold come from the reference only -- AoU samples never influence the ellipsoid, they're just scored against it.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

## Inputs

In [ ]:
REF_EIGENVEC_PATH = "PLACEHOLDER"   # 1kg_all_pca.eigenvec, from submit_pca_r1.ipynb
REF_PANEL_PATH = "PLACEHOLDER"      # integrated_call_samples_v3.20130502.ALL.panel
AOU_SSCORE_PATH = "PLACEHOLDER"     # acaf_projected.sscore, from submit_pca_r1.ipynb
OUT_DIR = "PLACEHOLDER"             # keep-list + diagnostic plots written here

N_PCS = 6                # matches the validated round-2-prototype parameters
THRESHOLD_QUANTILE = 0.9
REF_POPS = ["CEU", "GBR"]   # reference cluster the ellipsoid is fit to

import os
os.makedirs(OUT_DIR, exist_ok=True)

## Load reference PCs + population labels

In [ ]:
pc_cols = [f"PC{i}" for i in range(1, N_PCS + 1)]

ref = pd.read_csv(REF_EIGENVEC_PATH, sep=r"\s+")
ref_id_col = "#IID" if "#IID" in ref.columns else "IID"
ref = ref.rename(columns={ref_id_col: "sample"})

labels = pd.read_csv(REF_PANEL_PATH, sep="\t")
ref = ref.merge(labels[["sample", "pop", "super_pop"]], on="sample")

assert all(c in ref.columns for c in pc_cols), f"missing PC columns, have: {list(ref.columns)}"
print("Samples per population:")
print(ref["pop"].value_counts())

## Fit the ellipsoid to the reference cluster

In [ ]:
ref_mask = ref["pop"].isin(REF_POPS)
ref_pcs = ref.loc[ref_mask, pc_cols].values
assert len(ref_pcs) > N_PCS, f"too few reference samples ({len(ref_pcs)}) to fit a {N_PCS}-PC covariance"

ref_mean = ref_pcs.mean(axis=0)
ref_cov = np.cov(ref_pcs, rowvar=False)
ref_cov_inv = np.linalg.inv(ref_cov)

threshold = np.sqrt(chi2.ppf(THRESHOLD_QUANTILE, df=N_PCS))
print(f"Mahalanobis threshold ({N_PCS} PCs, {THRESHOLD_QUANTILE:.0%}): {threshold:.3f}")


def mahal(x, mean, cov_inv):
    d = x - mean
    return np.sqrt(d @ cov_inv @ d)

## Validate against the full reference panel

Sanity check before touching AoU data -- confirms the fitted ellipsoid actually separates the reference cluster from other reference populations, same check as the earlier chr22 prototype.

In [ ]:
ref["mahal"] = [mahal(row, ref_mean, ref_cov_inv) for row in ref[pc_cols].values]
ref["pass"] = ref["mahal"] <= threshold

print("Pass rate by population:")
summary = ref.groupby("pop")["pass"].agg(["sum", "count"])
summary["pct"] = (summary["sum"] / summary["count"] * 100).round(1)
print(summary.sort_values("pct", ascending=False))

ref_retention = ref.loc[ref_mask, "pass"].mean()
print(f"\n{'+'.join(REF_POPS)} retention: {ref_retention:.1%}")

## Score AoU samples against the same ellipsoid

`AOU_SSCORE_PATH` should already be on the same scale as the reference's own eigenvectors -- `submit_pca_r1.ipynb`'s projection step uses `--score ... sum`, matching the reference PCA's own convention. Column names here are `PC{i}_SUM`, not `PC{i}`.

In [ ]:
aou = pd.read_csv(AOU_SSCORE_PATH, sep=r"\s+")
aou_id_col = "#IID" if "#IID" in aou.columns else "IID"
aou = aou.rename(columns={aou_id_col: "sample"})

aou_pc_cols = [f"PC{i}_SUM" for i in range(1, N_PCS + 1)]
assert all(c in aou.columns for c in aou_pc_cols), f"missing PC columns, have: {list(aou.columns)}"

aou["mahal"] = [mahal(row, ref_mean, ref_cov_inv) for row in aou[aou_pc_cols].values]
aou["pass"] = aou["mahal"] <= threshold

n_pass = aou["pass"].sum()
print(f"AoU samples: {len(aou)}")
print(f"Passing round 1 filter: {n_pass} ({n_pass / len(aou):.1%})")

## Write the keep-list

AoU person IDs only -- participant-level, stays in the bucket, never committed to git.

In [ ]:
keep_path = os.path.join(OUT_DIR, "round1_keep_ids.txt")
aou.loc[aou["pass"], "sample"].to_csv(keep_path, index=False, header=False)
print(f"Wrote {n_pass} IDs to {keep_path}")

## Diagnostics

PC1 vs PC2 with the fitted ellipse, reference populations vs AoU overlaid; Mahalanobis distance distributions. Plots saved under `OUT_DIR`, not committed.

In [ ]:
colors = {
    "CEU": "#2166ac", "GBR": "#4dac26", "FIN": "#d01c8b",
    "TSI": "#f1a340", "IBS": "#998ec3",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for pop, grp in ref.groupby("pop"):
    axes[0].scatter(grp["PC1"], grp["PC2"], c=colors.get(pop, "grey"), label=pop, alpha=0.6, s=15)

axes[0].scatter(
    aou["PC1_SUM"], aou["PC2_SUM"],
    c=np.where(aou["pass"], "black", "red"),
    marker="x", alpha=0.3, s=10, label="AoU",
)

cov_2d = np.cov(ref_pcs[:, :2], rowvar=False)
mean_2d = ref_pcs[:, :2].mean(axis=0)
eigvals, eigvecs = np.linalg.eigh(cov_2d)
angle = np.degrees(np.arctan2(*eigvecs[:, 1][::-1]))
scale_2d = np.sqrt(chi2.ppf(THRESHOLD_QUANTILE, df=2))
width = 2 * scale_2d * np.sqrt(eigvals[1])
height = 2 * scale_2d * np.sqrt(eigvals[0])
ellipse = Ellipse(
    xy=mean_2d, width=width, height=height, angle=angle,
    fill=False, edgecolor="black", linewidth=2, linestyle="--",
    label=f"{THRESHOLD_QUANTILE:.0%} {'+'.join(REF_POPS)} ellipsoid (2D slice)",
)
axes[0].add_patch(ellipse)
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].set_title("PC1 vs PC2 -- reference + AoU")
axes[0].legend(markerscale=2, fontsize=8)

for pop, grp in ref.groupby("pop"):
    axes[1].hist(grp["mahal"], bins=30, alpha=0.5, label=pop, color=colors.get(pop, "grey"), density=True)
axes[1].hist(aou["mahal"], bins=30, alpha=0.4, label="AoU", color="black", density=True)
axes[1].axvline(threshold, color="black", linestyle="--", linewidth=2, label=f"Threshold ({threshold:.2f})")
axes[1].set_xlabel(f"Mahalanobis distance from {'+'.join(REF_POPS)} centroid ({N_PCS} PCs)")
axes[1].set_ylabel("Density")
axes[1].legend(fontsize=8)

plt.suptitle(
    f"Round 1 ancestry filter -- {'+'.join(REF_POPS)} retention: {ref_retention:.1%}  |  "
    f"AoU pass rate: {n_pass / len(aou):.1%}",
    fontsize=11,
)
plt.tight_layout()
plot_path = os.path.join(OUT_DIR, "round1_filter.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved to {plot_path}")